### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### RAG Pipeline

In [3]:
client = instructor.from_provider(
    "openai/gpt-6-luna",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [4]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [5]:
qdrant_client = QdrantClient("url=http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="toys_and_games_small_sample",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["all_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, query):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response =  client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(query, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [6]:
output = rag_pipeline("Any USB chargeable toy for girl?", qdrant_client)

In [7]:
output

{'data_object': RAGGenerationResponse(answer='The available products don’t clearly list a USB-chargeable toy specifically for girls. The Kidstinct RC tank includes a USB charger and rechargeable batteries, and has a bubble turret, but it’s marketed for kids generally. The butterfly bubble machine is suitable for girls but requires 3 AA batteries and isn’t USB-chargeable.'),
 'answer': 'The available products don’t clearly list a USB-chargeable toy specifically for girls. The Kidstinct RC tank includes a USB charger and rechargeable batteries, and has a bubble turret, but it’s marketed for kids generally. The butterfly bubble machine is suitable for girls but requires 3 AA batteries and isn’t USB-chargeable.',
 'question': 'Any USB chargeable toy for girl?',
 'retrieved_context_ids': ['B09JKLK1DG',
  'B09HXCRK75',
  'B0BLHHD8MZ',
  'B0BFB9N1JY',
  'B0BC8TJGQP'],
 'retrieved_context': ['Butterfly Bubble Machine for Toddlers,Magic Bubble Wand for Kids,Musical Light Up Bubble Blower with 2

In [8]:
print(output["answer"])

The available products don’t clearly list a USB-chargeable toy specifically for girls. The Kidstinct RC tank includes a USB charger and rechargeable batteries, and has a bubble turret, but it’s marketed for kids generally. The butterfly bubble machine is suitable for girls but requires 3 AA batteries and isn’t USB-chargeable.


### RAG Pipeline with Grounding Context

In [9]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [10]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="toys_and_games_small_sample",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["all_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [14]:
output = rag_pipeline("Do you have any cars?")

In [15]:
output

{'data_object': RAGGenerationResponse(answer='Yes, the available products include these cars and vehicles:\n- Disney Pixar Cars 3 Junior Moon diecast vehicle (B0718T5BXP)\n- 2019 Ford Mustang RC drift car (B08V616T5G)\n- Hot Wheels Bone Shaker monster truck (B09G7K3JWQ)\n- Marvel Deadpool Taco Truck (B07JYJNTJ8)\n- Back to the Future DeLorean time machine (B0021L8FM6)', references=[RAGUsedContext(id='B0718T5BXP', description='Disney Cars 3 Junior Moon diecast vehicle'), RAGUsedContext(id='B08V616T5G', description='Jada Toys 2019 Ford Mustang RC drift car'), RAGUsedContext(id='B09G7K3JWQ', description='Hot Wheels 1:24 Bone Shaker monster truck'), RAGUsedContext(id='B07JYJNTJ8', description='Jada Marvel Deadpool Taco Truck'), RAGUsedContext(id='B0021L8FM6', description='Diamond Select Back to the Future DeLorean')]),
 'answer': 'Yes, the available products include these cars and vehicles:\n- Disney Pixar Cars 3 Junior Moon diecast vehicle (B0718T5BXP)\n- 2019 Ford Mustang RC drift car (B

In [16]:
print(output["answer"])

Yes, the available products include these cars and vehicles:
- Disney Pixar Cars 3 Junior Moon diecast vehicle (B0718T5BXP)
- 2019 Ford Mustang RC drift car (B08V616T5G)
- Hot Wheels Bone Shaker monster truck (B09G7K3JWQ)
- Marvel Deadpool Taco Truck (B07JYJNTJ8)
- Back to the Future DeLorean time machine (B0021L8FM6)


In [18]:
output = rag_pipeline("Do you have any cars?", top_k=10)

In [19]:
output

{'data_object': RAGGenerationResponse(answer='Yes—available cars include:\n- Disney Pixar Cars 3 Junior Moon die-cast vehicle (1:55 scale)\n- Jada Toys Ford Mustang drift RC car (1:10 scale)\n- Hot Wheels Bone Shaker monster truck (1:24 scale)\n- Jada Toys Marvel Deadpool Taco Truck (1:32 scale)\n- Back to the Future DeLorean replica (1:18 scale, with lights and sounds)\n- Kidstinct RC stunt tank/car with drifting, water-bullet, and bubble features', references=[RAGUsedContext(id='B0718T5BXP', description='Disney Cars 3 Junior Moon die-cast vehicle, 1:55 scale'), RAGUsedContext(id='B08V616T5G', description='Jada Toys 2019 Ford Mustang drift RC, 1:10 scale'), RAGUsedContext(id='B09G7K3JWQ', description='Hot Wheels 2022 Bone Shaker monster truck, 1:24 scale'), RAGUsedContext(id='B07JYJNTJ8', description='Jada Toys Marvel Deadpool Taco Truck, 1:32 scale'), RAGUsedContext(id='B0021L8FM6', description='Back to the Future DeLorean replica, 1:18 scale, with working lights and sounds'), RAGUse